# Real-time prediction and posthoc analysis

This notebook loads a trained checkpoint, streams an arbitrary-length MP150 trial, predicts online, then labels the saved raw trial from the audio channel for posthoc scoring.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / 'testing.py').exists():
    ROOT = Path(r'D:/BME/BCI/online_bci/online_eeg')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from acquisition import AcquisitionConfig
from preprocessing import AudioLabelConfig, PreprocessConfig, preprocess_recording
from testing import (
    posthoc_analyze_realtime_trial,
    run_realtime_mp150_prediction,
    evaluate_prediction_log_against_labeled_recording,
)
from plots import plot_labeled_recording, plot_predictions_overlay

print('Pipeline root:', ROOT)


## Settings

In [ ]:
RUN_ID = 'run_001'
RUN_DIR = ROOT / 'runs' / RUN_ID
TEST_DIR = RUN_DIR / 'realtime_trials'
CHECKPOINT_PATH = RUN_DIR / 'lstm_checkpoint.pt'  # Saved by the training notebook.

COLLECT_REALTIME = True
TRIAL_NAME = 'realtime_trial_01'
TRIAL_SEC = 300.0
PRINT_EVERY_PREDICTION = True
PREDICTION_FLUSH_EVERY = 10  # Flush prediction CSV every N rows; set 0/None to rely on normal buffering.

EXISTING_RAW_NPZ = None
EXISTING_PREDICTION_CSV = None
EXISTING_LABELED_NPZ = None

EEG_CHANNELS = (1, 2, 3, 4)
EEG_CHANNEL_NAMES = ('O1', 'Oz', 'O2', 'POz')
AUDIO_CHANNEL = 16
APPLY_SOFTWARE_FILTERS = False  # BIOPAC hardware already bandpasses the EEG at 1-35 Hz.
SOFTWARE_BANDPASS_HZ = None  # Set to (1.0, 35.0) only if APPLY_SOFTWARE_FILTERS=True.
SOFTWARE_NOTCH_HZ = None  # Set to (60.0,) only if APPLY_SOFTWARE_FILTERS=True.

ACQ = AcquisitionConfig(
    samplerate=200,
    channels=(*EEG_CHANNELS, AUDIO_CHANNEL),
    chunk_sec=0.05,
)
REALTIME_STRIDE_SEC = ACQ.chunk_sec

PRE = PreprocessConfig(
    eeg_channels=EEG_CHANNELS,
    audio_channel=AUDIO_CHANNEL,
    apply_software_filters=APPLY_SOFTWARE_FILTERS,
    bandpass_low_hz=None if SOFTWARE_BANDPASS_HZ is None else SOFTWARE_BANDPASS_HZ[0],
    bandpass_high_hz=None if SOFTWARE_BANDPASS_HZ is None else SOFTWARE_BANDPASS_HZ[1],
    notch_hz=SOFTWARE_NOTCH_HZ,
    notch_quality_factor=30.0,
    filter_order=4,
    demean_channels=True,
)

LABELS = AudioLabelConfig(
    class_names=('Eyes Open', 'Eyes Closed'),
    baseline_label=0,
    active_label=1,
    cue_label_sequence=None,
    alternate_binary_labels=True,
    label_duration_sec=None,  # transition mode: each cue switches state until the next cue.
    label_start_offset_sec=0.0,  # label switch starts exactly at cue onset.
    envelope_window_sec=0.025,
    onset_threshold=None,
    onset_min_interval_sec=0.50,
)

RUN_DIR.mkdir(parents=True, exist_ok=True)
TEST_DIR.mkdir(parents=True, exist_ok=True)
print('Run directory:', RUN_DIR)


## Stream trial and predict online

In [ ]:
if COLLECT_REALTIME:
    realtime_result = run_realtime_mp150_prediction(
        output_dir=TEST_DIR,
        checkpoint_path=CHECKPOINT_PATH,
        acquisition_config=ACQ,
        preprocess_config=PRE,
        duration_sec=TRIAL_SEC,
        trial_name=TRIAL_NAME,
        print_every_prediction=PRINT_EVERY_PREDICTION,
        prediction_stride_sec=REALTIME_STRIDE_SEC,
        prediction_flush_every=PREDICTION_FLUSH_EVERY,
    )
    raw_npz = realtime_result['raw_path']
    prediction_csv = realtime_result['prediction_csv']
    aligned_prediction_csv = realtime_result.get('aligned_prediction_csv')
else:
    raw_npz = Path(EXISTING_RAW_NPZ) if EXISTING_RAW_NPZ is not None else None
    prediction_csv = Path(EXISTING_PREDICTION_CSV) if EXISTING_PREDICTION_CSV is not None else None
    aligned_prediction_csv = None

if raw_npz is None or prediction_csv is None:
    raise ValueError('Set COLLECT_REALTIME=True or provide EXISTING_RAW_NPZ and EXISTING_PREDICTION_CSV.')

print('Prediction CSV:', prediction_csv)
if aligned_prediction_csv is not None:
    print('Realtime aligned EEG/prediction CSV:', aligned_prediction_csv)
pd.read_csv(prediction_csv).tail()


## Posthoc audio-labeling and scoring

In [ ]:
if EXISTING_LABELED_NPZ is None:
    analysis = posthoc_analyze_realtime_trial(
        raw_npz=raw_npz,
        prediction_csv=prediction_csv,
        checkpoint_path=CHECKPOINT_PATH,
        output_dir=TEST_DIR,
        preprocess_config=PRE,
        label_config=LABELS,
    )
    labeled_npz = analysis['labeled_path']
else:
    labeled_npz = Path(EXISTING_LABELED_NPZ)
    analysis = evaluate_prediction_log_against_labeled_recording(
        prediction_csv=prediction_csv,
        labeled_npz=labeled_npz,
        output_dir=TEST_DIR,
        checkpoint_path=CHECKPOINT_PATH,
    )

print('Evaluated prediction CSV:', analysis['evaluated_csv'])
print('Aligned EEG/prediction CSV:', analysis['aligned_prediction_csv'])
display(analysis['summary'])
display(analysis['per_class'])
display(analysis['delay_summary'])
display(analysis['cue_delay_summary'])
display(analysis['xcov_delay_summary'])


In [ ]:
fig, axes = plot_labeled_recording(
    labeled_npz,
    max_duration_sec=30.0,
    channel_names=EEG_CHANNEL_NAMES,
)


In [ ]:
fig, axes = plot_predictions_overlay(
    labeled_npz=labeled_npz,
    predictions=analysis['evaluated_predictions'],
    max_duration_sec=None,
    channel_names=EEG_CHANNEL_NAMES,
    show_true_labels=True,
)
